## Part 1 - Inspect the dataset
Each row represents one penguin observation. species and island are categorical variables; bill_length_mm measures bill length in mm and body_mass_g measures body mass in g. year is the numeric observation year, rather than a continuous body measurement.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import hashlib
import sys
pd.set_option('display.width', 100)
pd.set_option('display.max_columns', 8)
source = Path('penguins.csv')
source_hash = hashlib.sha256(source.read_bytes()).hexdigest()
penguins = pd.read_csv(source, na_values=['NA'])
original = penguins.copy(deep=True)
print('Python:', sys.version.split()[0], '| pandas:', pd.__version__)
print(penguins.head(6).to_string(index=False))
print('Dimensions:', penguins.shape)
print('Types:\n', penguins.dtypes.to_string())
print('Missing values:\n', penguins.isna().sum().to_string())
assert penguins.shape == (344, 8)
print('PASS: 344 rows, 8 columns')

Python: 3.12.14 | pandas: 3.0.1
species    island  bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g    sex  year
 Adelie Torgersen            39.1           18.7              181.0       3750.0   male  2007
 Adelie Torgersen            39.5           17.4              186.0       3800.0 female  2007
 Adelie Torgersen            40.3           18.0              195.0       3250.0 female  2007
 Adelie Torgersen             NaN            NaN                NaN          NaN    NaN  2007
 Adelie Torgersen            36.7           19.3              193.0       3450.0 female  2007
 Adelie Torgersen            39.3           20.6              190.0       3650.0   male  2007
Dimensions: (344, 8)
Types:
 species                  str
island                   str
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                      str
year                   int64
Missing values:
 species               0
island  

### Python/R comparison - inferred types
Python pandas reads flipper_length_mm and body_mass_g as float64 because ordinary NumPy integers cannot represent NaN. R reads these whole-number measurements as integer even with NA. year is int64 in Python and integer in R. Text columns are str in the pandas version used here and character in R; categorical meaning does not automatically imply a category/factor type. Decimal bill measurements are float64/double. Both implementations have identical dimensions and missing-value counts.

## Part 2 - Select rows and columns
Observations 5 through 10 refer to positions in the original file. Labels p01-p06 are illustrative and are not actual penguin identifiers.

In [2]:
cols = ['species', 'island', 'flipper_length_mm']
selected = penguins.loc[:, cols].copy()
vector = penguins['flipper_length_mm']
one_column = penguins[['flipper_length_mm']]
print('Selection:', selected.shape)
print('1D:', type(vector).__name__, vector.shape, 'size:', vector.size)
print('Table:', type(one_column).__name__, one_column.shape, 'size:', one_column.size)
positions = selected.iloc[4:10]
print(positions.to_string(index=False))
assert selected.shape == (344, 3) and positions.shape == (6, 3)
assert vector.ndim == 1 and one_column.ndim == 2
small = selected.iloc[:6].copy()
small.index = [f'p{i:02d}' for i in range(1, 7)]
assert small.index.is_unique
print('Selection by labels:\n', small.loc[['p02', 'p05']].to_string())
reverse = small.iloc[::-1]
print('Reversed table:\n', reverse.to_string())
print('First row by position:\n', reverse.iloc[[0]].to_string())
print('p01 by label:\n', reverse.loc[['p01']].to_string())
assert reverse.index[0] == 'p06'
assert reverse.loc[['p01']].equals(small.loc[['p01']])
print('PASS: dimensions, unique labels and selections')

Selection: (344, 3)
1D: Series (344,) size: 344
Table: DataFrame (344, 1) size: 344
species    island  flipper_length_mm
 Adelie Torgersen              193.0
 Adelie Torgersen              190.0
 Adelie Torgersen              181.0
 Adelie Torgersen              195.0
 Adelie Torgersen              193.0
 Adelie Torgersen              190.0
Selection by labels:
     species     island  flipper_length_mm
p02  Adelie  Torgersen              186.0
p05  Adelie  Torgersen              193.0
Reversed table:
     species     island  flipper_length_mm
p06  Adelie  Torgersen              190.0
p05  Adelie  Torgersen              193.0
p04  Adelie  Torgersen                NaN
p03  Adelie  Torgersen              195.0
p02  Adelie  Torgersen              186.0
p01  Adelie  Torgersen              181.0
First row by position:
     species     island  flipper_length_mm
p06  Adelie  Torgersen              190.0
p01 by label:
     species     island  flipper_length_mm
p01  Adelie  Torgersen           

After reversing the rows, p06 occupies the first position, while p01 still identifies the original first row. Positional selection follows order; label selection follows identity. Python positions start at 0 and the slice endpoint is excluded; R positions start at 1. In R, preserve a table with drop = FALSE or with penguins["flipper_length_mm"].

## Part 3 - Filter observations
Each filter starts independently from penguins. Missing measurements are explicitly excluded. For readability, the first ten retained rows are displayed; complete results are stored in f1, f2 and f3. Checks cover every retained row.

In [3]:
mask1 = penguins.flipper_length_mm.notna() & (penguins.flipper_length_mm >= 210)
mask2 = ((penguins.species == 'Gentoo') & penguins.body_mass_g.notna()
         & (penguins.body_mass_g >= 5000))
mask3 = ((penguins.island.isin(['Biscoe', 'Dream']))
         & penguins.bill_length_mm.notna() & (penguins.bill_length_mm < 45))
f1 = penguins.loc[mask1].copy()
f2 = penguins.loc[mask2].copy()
f3 = penguins.loc[mask3].copy()
for label, result, relevant in [
    ('F1', f1, ['species', 'island', 'flipper_length_mm']),
    ('F2', f2, ['species', 'body_mass_g']),
    ('F3', f3, ['island', 'bill_length_mm'])]:
    print(label, 'count:', len(result))
    print(result[relevant].head(10).to_string(index=False))
assert f1.flipper_length_mm.notna().all() and (f1.flipper_length_mm >= 210).all()
assert (f2.species == 'Gentoo').all() and f2.body_mass_g.notna().all()
assert (f2.body_mass_g >= 5000).all()
assert f3.island.isin(['Biscoe', 'Dream']).all()
assert f3.bill_length_mm.notna().all() and (f3.bill_length_mm < 45).all()
filter_counts = [len(f1), len(f2), len(f3)]
print('PASS: every retained row meets the conditions', filter_counts)

F1 count: 114
species    island  flipper_length_mm
 Adelie Torgersen              210.0
 Gentoo    Biscoe              211.0
 Gentoo    Biscoe              230.0
 Gentoo    Biscoe              210.0
 Gentoo    Biscoe              218.0
 Gentoo    Biscoe              215.0
 Gentoo    Biscoe              210.0
 Gentoo    Biscoe              211.0
 Gentoo    Biscoe              219.0
 Gentoo    Biscoe              215.0
F2 count: 67
species  body_mass_g
 Gentoo       5700.0
 Gentoo       5700.0
 Gentoo       5400.0
 Gentoo       5200.0
 Gentoo       5150.0
 Gentoo       5550.0
 Gentoo       5850.0
 Gentoo       5850.0
 Gentoo       6300.0
 Gentoo       5350.0
F3 count: 127
island  bill_length_mm
Biscoe            37.8
Biscoe            37.7
Biscoe            35.9
Biscoe            38.2
Biscoe            38.8
Biscoe            35.3
Biscoe            40.6
Biscoe            40.5
Biscoe            37.9
Biscoe            40.5
PASS: every retained row meets the conditions [114, 67, 127]


In Python, comparing float NaN with a threshold returns False; in R, comparing NA returns NA. Both workflows therefore explicitly require a known measurement. With this handling, all three filter counts agree between Python and R; this agreement is also checked when building the reports.

## Part 4 - Build a working table
The conversion is vectorized: grams / 1000. Species is sorted ascending, mass descending, and missing masses last within each species. When both sorting keys tie, an explicit auxiliary position key preserves the original order. This auxiliary key is not retained in working.

In [4]:
working = penguins[['species', 'island', 'flipper_length_mm', 'body_mass_g']].copy()
working['body_mass_kg'] = working.body_mass_g / 1000
working['_source_row'] = np.arange(len(working))
working = working.sort_values(
    ['species', 'body_mass_g', '_source_row'], ascending=[True, False, True],
    na_position='last').drop(columns='_source_row').reset_index(drop=True)
print(working.head(10).to_string(index=False))
known = working.body_mass_g.notna()
assert np.allclose(working.loc[known, 'body_mass_kg'],
                   working.loc[known, 'body_mass_g'] / 1000, rtol=0, atol=1e-12)
assert len(working) == len(penguins)
assert working.body_mass_g.isna().sum() == penguins.body_mass_g.isna().sum()
assert penguins.shape[1] == 8 and working.shape[1] == 5
assert working.species.is_monotonic_increasing
for _, group in working.groupby('species', sort=False):
    assert group.body_mass_g.dropna().is_monotonic_decreasing
    assert group.body_mass_g.isna().is_monotonic_increasing
pd.testing.assert_frame_equal(penguins, original)
print('PASS: conversion (atol=1e-12), sorting, 344 rows, 5 columns,',
      working.body_mass_g.isna().sum(), 'missing masses; original unchanged')

species    island  flipper_length_mm  body_mass_g  body_mass_kg
 Adelie    Biscoe              197.0       4775.0         4.775
 Adelie    Biscoe              203.0       4725.0         4.725
 Adelie Torgersen              196.0       4700.0         4.700
 Adelie Torgersen              195.0       4675.0         4.675
 Adelie     Dream              184.0       4650.0         4.650
 Adelie     Dream              190.0       4600.0         4.600
 Adelie    Biscoe              191.0       4600.0         4.600
 Adelie Torgersen              197.0       4500.0         4.500
 Adelie     Dream              199.0       4475.0         4.475
 Adelie Torgersen              198.0       4450.0         4.450
PASS: conversion (atol=1e-12), sorting, 344 rows, 5 columns, 2 missing masses; original unchanged


## Part 5 - Export and verify
Exports omit row labels and write missing values as literal NA. We check dimensions, column names, types, missing values and all values, using a tolerance of 1e-12 for numerical comparisons. Separate filenames allow both languages to keep their exports in the same directory.

In [5]:
def verify_restored(restored):
    print('Dimensions:', restored.shape)
    print('Types:\n', restored.dtypes.to_string())
    print('Missing values:\n', restored.isna().sum().to_string())
    assert restored.shape == working.shape
    assert restored.columns.tolist() == working.columns.tolist()
    assert restored.isna().equals(working.isna())
    for column in working:
        if pd.api.types.is_numeric_dtype(working[column]):
            assert pd.api.types.is_numeric_dtype(restored[column])
            assert np.allclose(working[column], restored[column],
                               rtol=0, atol=1e-12, equal_nan=True)
        else:
            pd.testing.assert_series_equal(working[column], restored[column])
    assert restored.dtypes.equals(working.dtypes)
    print('PASS: dimensions, names, types, missing values and all values')

working.to_csv('working_python.csv', index=False, na_rep='NA')
restored = pd.read_csv('working_python.csv', na_values=['NA'])
verify_restored(restored)
working.to_csv('working_python_semicolon.csv', index=False, sep=';',
               decimal=',', na_rep='NA')
restored_semicolon = pd.read_csv('working_python_semicolon.csv', sep=';',
                                 decimal=',', na_values=['NA'])
verify_restored(restored_semicolon)

Dimensions: (344, 5)
Types:
 species                  str
island                   str
flipper_length_mm    float64
body_mass_g          float64
body_mass_kg         float64
Missing values:
 species              0
island               0
flipper_length_mm    2
body_mass_g          2
body_mass_kg         2
PASS: dimensions, names, types, missing values and all values
Dimensions: (344, 5)
Types:
 species                  str
island                   str
flipper_length_mm    float64
body_mass_g          float64
body_mass_kg         float64
Missing values:
 species              0
island               0
flipper_length_mm    2
body_mass_g          2
body_mass_kg         2
PASS: dimensions, names, types, missing values and all values


In [6]:
try:
    wrong = pd.read_csv('working_python_semicolon.csv')
    print('Incorrect import - dimensions:', wrong.shape)
    print('Columns:', wrong.columns.tolist())
    print(wrong.head(3).to_string())
    assert wrong.shape != working.shape or wrong.columns.tolist() != working.columns.tolist()
except (pd.errors.ParserError, ValueError) as error:
    print('Expected parser error:', type(error).__name__, str(error)[:250])
# The correction uses the same parameters as the successful import above.
corrected_import = pd.read_csv('working_python_semicolon.csv', sep=';',
                               decimal=',', na_values=['NA'])
verify_restored(corrected_import)
assert hashlib.sha256(source.read_bytes()).hexdigest() == source_hash
pd.testing.assert_frame_equal(penguins, original)
print('PASS: the source CSV and penguins object are unchanged')
print('Filter counts:', filter_counts)

Incorrect import - dimensions: (344, 1)
Columns: ['species;island;flipper_length_mm;body_mass_g;body_mass_kg']
                                 species;island;flipper_length_mm;body_mass_g;body_mass_kg
Adelie;Biscoe;197    0;4775 0;4                                                      775.0
Adelie;Biscoe;203    0;4725 0;4                                                      725.0
Adelie;Torgersen;196 0;4700 0;4                                                        7.0
Dimensions: (344, 5)
Types:
 species                  str
island                   str
flipper_length_mm    float64
body_mass_g          float64
body_mass_kg         float64
Missing values:
 species              0
island               0
flipper_length_mm    2
body_mass_g          2
body_mass_kg         2
PASS: dimensions, names, types, missing values and all values
PASS: the source CSV and penguins object are unchanged
Filter counts: [114, 67, 127]


### CSV interpretation and submission preparation
The default comma-separated parser does not recognize semicolons and mistakes decimal commas for separators. This produces an incorrect table or a parser error; expected errors are handled so execution can continue. The correction uses sep=";" and decimal="," in Python, or dec="," in R.
CSV preserves textual representations of values, column names and row/column order. It does not preserve data types, units, categories/factors, index/row labels or notebook metadata. In these exports, missing values are recognized using the agreed NA string. Other datasets may need to distinguish literal text "NA" from a missing value. Types are inferred again and must be verified or specified explicitly.
Python/R comparison: both versions have 344 x 8 in the source and 344 x 5 in working, identical filters, preserved missing values and equal values after both exports. Python uses Series/DataFrame; R uses vector/data.frame and drop=FALSE. R additionally demonstrates how indexing with an NA mask differs from subset().
The notebooks and PDFs were generated from code executed in fresh local processes. To complete the required Colab step, open each notebook, select the Python 3 / R runtime, upload penguins.csv, restart the session and run all cells. Then export the report to PDF again if needed. The ZIP contains exactly two PDF reports; CSV files remain in the working directory.